# Globally Injective Parametrization
Demonstrates flexibility of MeshFEM's IPC integration by implementing a globally injective parametrization using a locally injective parametrization energy and an IPC self-collision barrier energy.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [ ]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, param_utils, benchmark
import energy
import numpy as np

m = mesh.Mesh('../models/hand.msh')
# m = mesh.Mesh('../models/cow2Disc.msh')

In [ ]:
# m = param_utils.load('../models/hilbert_curve.msh.xz')

In [ ]:
m = param_utils.load('../models/lucy.msh.xz')

In [ ]:
# m = param_utils.load('../models/bird.msh.xz')

In [ ]:
# Scale so that the surface area is pi (to match [Su et al. 2020])
m.setVertices(m.vertices() * np.sqrt(np.pi / m.volume))

In [ ]:
INCLUDE_CONTACT = False
RECORD_VIDEO = False

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
uv_init = param_utils.tutteInitialization(m)
uv.setVars(uv_init.ravel())

In [ ]:
e = energy.SymmetricDirichlet(2)
param = mesh_energy.Parametrization(m, uv, e)
objectives = [param]

In [ ]:
# Globally scale to minimize energy
# import dirichlet_demo
# a = dirichlet_demo.param_dirichlet_edensity(m, uv).objective()
# b = param.objective() - a
# s = (b / a)**(1/4)
# print(s)
# uv.setVars(s * uv_init.ravel())

In [ ]:
# (manually) globally scale to minimize gradient norm
import dirichlet_demo
uv.setVars(298 * uv_init.ravel())
np.linalg.norm(param.gradient())

In [ ]:
# Use original scale (CM)
# import dirichlet_demo
# uv.setVars(uv_init.ravel())
# np.linalg.norm(param.gradient())

In [ ]:
if INCLUDE_CONTACT:
    import meshfem_ipc
    cm = meshfem_ipc.CollisionMesh(m, embeddingDimension=2)
    contact = meshfem_ipc.IPCObjectiveTerm(uv, cm)
    contact.useAdaptiveBarrier = True
    objectives.append(contact)
    
    contact.ccdTol = 1e-3

In [ ]:
# contact.barrierStiffness = 1e5

In [ ]:
# contact.dhat = 0.0001

In [ ]:
# Construct parametrization energy and problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)

In [ ]:
import flip_avoiding_step_length
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.9

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
# Work around energy nullspace by adding a small shift
prob.hessianShift = 1e-12
prob.useRelativeHessianShift = True
opt = prob.optimizer()
opt.options.niter = 500

In [ ]:
param.useXBasedProjection = False

In [ ]:
opt.options.gradTol = 0.1

In [ ]:
prob.setCustomIterationCallback(v.updater(10))

In [ ]:
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
opt.options.hessianProjectionController.startWithProjectionActive = True

In [ ]:
# opt.options.factorizer = opt.options.factorizer.CatamariAMD

In [ ]:
if RECORD_VIDEO:
    prob.setCustomIterationCallback(v.updater())
    name = 'globally_injective' if INCLUDE_CONTACT else 'not_globally_injective'
    v.recordStart(f'{name}.mp4', renderScale=4, outputScale=2, lineWidthScale=0.5)

In [ ]:
benchmark.reset()
rep = opt.optimize()
benchmark.report()

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.semilogy(orig_rep.energy)
plt.semilogy(gscale_rep.energy)
plt.semilogy(rep.energy)

In [ ]:
plt.semilogy(orig_rep.freeGradientNorm)
plt.semilogy(gscale_rep.freeGradientNorm)
plt.semilogy(rep.freeGradientNorm)